# 06 Local-Region Expert MoE x DQA Fifteen Loops

This notebook tests the next MoE idea:

> Do not mix models at the client level.  Grow experts around local
> pseudo-GT regions that are actually learnable.

This is a fast design/evidence loop, not fifteen full YOLO trainings.
It uses the completed `03_main_bn_residual_dqa_experiment` and the
previous MoE router results to rank fifteen local-region expert
hypotheses, then writes the concrete next full experiment candidate.

## Research Seeds

- PSSFL/FedMox: spatial router + Soft-Mixture for practical
  semi-supervised federated object detection.
- Soft MoE: differentiable soft assignment instead of brittle hard
  token routing.
- Expert Choice Routing: experts choose fixed-capacity tokens, which
  maps naturally to pseudo-GT quota control.
- MMoE / PLE: shared-private experts for task/domain relationship
  modeling.
- DAMEX: detection can benefit from dataset/domain-aware MoE, but
  routing should avoid expert collapse.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "moe":
    MOE_ROOT = cwd.parent
elif (cwd / "dynamic_quality_aware_classwise_aggregation").exists():
    MOE_ROOT = cwd / "dynamic_quality_aware_classwise_aggregation" / "scene_daynight_dqa" / "moe"
else:
    MOE_ROOT = cwd

SCENE_ROOT = MOE_ROOT.parent
WORKSPACE = MOE_ROOT / "output" / "06_spatial_expert_fifteen_loops"
SOURCE_WORKSPACE = SCENE_ROOT / "output" / "03_main_bn_residual_dqa_experiment"
PREV_MOE_WORKSPACE = MOE_ROOT / "output" / "05_router_ten_loops"
RUNNER = MOE_ROOT / "scripts" / "run_moe_06_spatial_expert_fifteen_loops.py"

print("MOE_ROOT", MOE_ROOT)
print("SOURCE_WORKSPACE", SOURCE_WORKSPACE)
print("PREV_MOE_WORKSPACE", PREV_MOE_WORKSPACE)
print("WORKSPACE", WORKSPACE)
print("RUNNER", RUNNER)

## Execute Fifteen Screening Loops

In [ ]:
cmd = [
    sys.executable,
    str(RUNNER),
    "--workspace-root", str(WORKSPACE),
    "--source-workspace", str(SOURCE_WORKSPACE),
    "--prev-moe-workspace", str(PREV_MOE_WORKSPACE),
    "--notify",
]
print(" ".join(cmd))
subprocess.run(cmd, cwd=MOE_ROOT, check=True)

## Split Evidence

In [ ]:
split_evidence = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_split_evidence.csv")
display(split_evidence)

## Fifteen Loop Scoreboard

In [ ]:
scoreboard = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_scoreboard.csv")
display(
    scoreboard[
        [
            "loop_id",
            "rank_score",
            "screened_projected_map50_95",
            "screened_delta_map50_95",
            "confidence",
            "implementation_change",
            "rationale",
        ]
    ]
)

## Explicit Fifteen-loop Trace

In [ ]:
trace = pd.read_csv(WORKSPACE / "stats" / "06_spatial_expert_loop_trace.csv")
display(
    trace[
        [
            "loop_index",
            "loop_id",
            "step_1_research",
            "step_5_execution",
            "step_6_result_summary",
            "step_7_next_direction",
        ]
    ]
)

## Selected Full Experiment Candidate

In [ ]:
import json

candidate_path = WORKSPACE / "stats" / "06_selected_full_experiment_candidate.json"
candidate = json.loads(candidate_path.read_text(encoding="utf-8"))
print(json.dumps(candidate, indent=2, ensure_ascii=False))

## Markdown Report

In [ ]:
report = WORKSPACE / "06_spatial_expert_fifteen_loop_report.md"
print(report)
print(report.read_text(encoding="utf-8")[:7000])